In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import random

# ============================================================
# Paths
# ============================================================

PREF_MH_DIR = Path("files/terrarium_butterfly_pref_mcmc_sdxl_turbo_joint")
POINTWISE_MH_DIR = Path("files_pointwise/terrarium_butterfly_pointwise_mh_sdxl_turbo_joint")

# Baseline images are identical in both folders.
BASELINE_DIR = PREF_MH_DIR

OUT_PDF = "mh_image_grid_3methods_4samples.pdf"
OUT_PNG = "mh_image_grid__3methods_4samples.png"

# ============================================================
# Plot settings
# ============================================================

# Use exactly four samples per method.
# These step labels are not shown in the figure.
STEPS = [300, 1200, 1800, 2700]

RANDOM_SEED = 7

METHODS = [
    ("Pref-MH (ours)", PREF_MH_DIR, "trajectory"),
    ("Pointwise-MH", POINTWISE_MH_DIR, "trajectory"),
    ("Base", BASELINE_DIR, "baseline"),
]

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.size": 9,
    "axes.linewidth": 0.45,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Helpers
# ============================================================

def load_image(path: Path, image_size: int = 440) -> Image.Image:
    img = Image.open(path).convert("RGB")

    w, h = img.size
    side = max(w, h)

    padded = Image.new("RGB", (side, side), color="white")
    padded.paste(img, ((side - w) // 2, (side - h) // 2))

    return padded.resize((image_size, image_size), Image.LANCZOS)


def get_state_image(folder: Path, step: int) -> Path:
    # Your trajectory files use zero-indexed step names.
    step_idx = step - 1

    exact = folder / f"state_sample_{step_idx:04d}.png"
    if exact.exists():
        return exact

    matches = sorted(folder.glob(f"*{step_idx:04d}*.png"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find image for step {step_idx} in {folder}")


def get_baseline_files(folder: Path, n_samples: int, seed: int) -> list[Path]:
    baseline_pool = sorted(folder.glob("baseline_*.png"))

    if len(baseline_pool) == 0:
        raise FileNotFoundError(f"No baseline_*.png images found in {folder}")

    if len(baseline_pool) < n_samples:
        raise ValueError(
            f"Requested {n_samples} baseline images, but only found "
            f"{len(baseline_pool)} in {folder}."
        )

    rng = random.Random(seed)
    return rng.sample(baseline_pool, n_samples)


# ============================================================
# Collect images
# ============================================================

method_images = []

for method_name, method_dir, source_type in METHODS:
    if source_type == "baseline":
        files = get_baseline_files(
            folder=method_dir,
            n_samples=len(STEPS),
            seed=RANDOM_SEED,
        )
    elif source_type == "trajectory":
        files = [get_state_image(method_dir, step) for step in STEPS]
    else:
        raise ValueError(f"Unknown source_type: {source_type}")

    method_images.append((method_name, files))


# ============================================================
# Plot
# ============================================================

n_methods = len(method_images)

fig = plt.figure(figsize=(7.35, 2.45), dpi=300)

# Outer layout: one block per method.
outer = fig.add_gridspec(
    1,
    n_methods,
    width_ratios=[1, 1, 1],
    wspace=0.18,
)

for method_idx, (method_name, files) in enumerate(method_images):
    # Each method block has:
    #   rows 0-1: 2x2 image grid
    #   row 2: centered method label
    inner = outer[0, method_idx].subgridspec(
        3,
        2,
        height_ratios=[1.0, 1.0, 0.09],
        wspace=0.05,
        hspace=0.05,
    )

    # Images in 2x2 structure
    for i, file_path in enumerate(files):
        r = i // 2
        c = i % 2

        ax = fig.add_subplot(inner[r, c])

        ax.imshow(load_image(file_path, image_size=440))
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.32)
            spine.set_edgecolor("0.82")

    # Label axis spans both columns
    label_ax = fig.add_subplot(inner[2, :])
    label_ax.axis("off")

    is_pref_mh = method_name == "Pref-MH (ours)"

    label_ax.text(
        0.5,
        0.75,
        method_name,
        ha="center",
        va="center",
        fontsize=10.2,
        fontweight="bold" if is_pref_mh else "normal",
        transform=label_ax.transAxes,
    )

fig.subplots_adjust(
    left=0.0,
    right=1.0,
    top=1.0,
    bottom=0.0,
)

fig.savefig(OUT_PDF, bbox_inches="tight", pad_inches=0.0)
fig.savefig(OUT_PNG, bbox_inches="tight", pad_inches=0.0)

plt.show()

print(f"Saved: {OUT_PDF}")
print(f"Saved: {OUT_PNG}")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import random

# ============================================================
# Paths
# ============================================================

PREF_MH_DIR = Path("files/terrarium_butterfly_pref_mcmc_sdxl_turbo_joint")
POINTWISE_MH_DIR = Path("files_pointwise/terrarium_butterfly_pointwise_mh_sdxl_turbo_joint")

# Base images are identical in both folders.
BASELINE_DIR = PREF_MH_DIR

OUT_PDF = "mh_image_grid_appendix_trajectory.pdf"
OUT_PNG = "mh_image_grid_appendix_trajectory.png"

# ============================================================
# Plot settings
# ============================================================

# Generate steps: 300, 600, 900, ..., 3000
MAX_STEP = 3000
STEP_SIZE = 300
STEPS = list(range(STEP_SIZE, MAX_STEP + 1, STEP_SIZE))

RANDOM_SEED = 7

METHODS = [
    ("Base", BASELINE_DIR, "baseline"),
    ("Pointwise-MH", POINTWISE_MH_DIR, "trajectory"),
    ("Pref-MH", PREF_MH_DIR, "trajectory"),
]

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.size": 10,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Helpers
# ============================================================

def load_image(path: Path, image_size: int = 512) -> Image.Image:
    # Slightly higher resolution for the appendix since images are larger
    img = Image.open(path).convert("RGB")

    w, h = img.size
    side = max(w, h)

    padded = Image.new("RGB", (side, side), color="white")
    padded.paste(img, ((side - w) // 2, (side - h) // 2))

    return padded.resize((image_size, image_size), Image.LANCZOS)


def get_state_image(folder: Path, step: int) -> Path:
    step_idx = step - 1

    exact = folder / f"state_sample_{step_idx:04d}.png"
    if exact.exists():
        return exact

    matches = sorted(folder.glob(f"*{step_idx:04d}*.png"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find image for step {step_idx} in {folder}")


def get_baseline_files(folder: Path, n_samples: int, seed: int) -> list[Path]:
    baseline_pool = sorted(folder.glob("baseline_*.png"))

    if len(baseline_pool) == 0:
        raise FileNotFoundError(f"No baseline_*.png images found in {folder}")

    if len(baseline_pool) < n_samples:
        raise ValueError(
            f"Requested {n_samples} baseline images, but only found "
            f"{len(baseline_pool)} in {folder}."
        )

    rng = random.Random(seed)
    return rng.sample(baseline_pool, n_samples)


# ============================================================
# Collect images
# ============================================================

method_images = {}

for method_name, method_dir, source_type in METHODS:
    if source_type == "baseline":
        # Base method doesn't have "steps", so we just grab N random samples
        files = get_baseline_files(
            folder=method_dir,
            n_samples=len(STEPS),
            seed=RANDOM_SEED,
        )
    elif source_type == "trajectory":
        files = [get_state_image(method_dir, step) for step in STEPS]
    else:
        raise ValueError(f"Unknown source_type: {source_type}")

    method_images[method_name] = files


# ============================================================
# Plot
# ============================================================

n_rows = len(STEPS)
n_cols = len(METHODS)

# Base width on standard text width (e.g., 5.5 inches).
# Calculate height proportionally to keep images perfectly square.
# (5.5 width / 3 cols) * 10 rows ≈ 18.3 inches tall.
fig_width = 5.5
fig_height = (fig_width / n_cols) * n_rows

fig, axes = plt.subplots(
    nrows=n_rows,
    ncols=n_cols,
    figsize=(fig_width, fig_height),
    dpi=300,
    gridspec_kw={"wspace": 0.04, "hspace": 0.04} # Tight spacing to eliminate whitespace
)

for row_idx, step in enumerate(STEPS):
    for col_idx, (method_name, _, _) in enumerate(METHODS):
        ax = axes[row_idx, col_idx]
        file_path = method_images[method_name][row_idx]

        ax.imshow(load_image(file_path))
        ax.set_xticks([])
        ax.set_yticks([])

        # Add a subtle border around each image
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_edgecolor("0.7")

        # Set Column Titles on the very first row
        if row_idx == 0:
            ax.set_title(method_name, fontsize=11, pad=8)

        # Set Row Labels (Steps) on the far left column
        if col_idx == 0:
            # We add a label to the left of the image.
            # For "Base", it's technically just a sample, but labeling the row
            # as "Step X" is standard for comparing against trajectories.
            ax.set_ylabel(f"Step {step}", fontsize=10, labelpad=8)

# Remove all outer margins
fig.subplots_adjust(
    left=0.05,  # Slight left margin to fit the 'Step X' text
    right=0.99,
    top=0.97,   # Slight top margin for column titles
    bottom=0.01,
)

# Use pad_inches=0.02 to crop tightly while keeping labels intact
fig.savefig(OUT_PDF, bbox_inches="tight", pad_inches=0.02)
fig.savefig(OUT_PNG, bbox_inches="tight", pad_inches=0.02)

plt.show()

print(f"Saved: {OUT_PDF}")
print(f"Saved: {OUT_PNG}")